# ICIMCIS 2026 — Benchmark V3: 15 paired repetitions and explicit write durability

**IMPORTANT:** This notebook conducts **new measurements**. It does not contain results from a full run. `smoke` output confirms basic execution only, never statistical significance.

### Updated after V2 smoke test
- **15 independent per-condition replicate runs** in full mode, with data restored before each database trial (vs 10 in V2). Each of the 60 volume × concurrency × operation comparisons has 15 paired repeats.
- **Durability explicitly configured and verified:** MySQL InnoDB `innodb_flush_log_at_trx_commit=1` and `sync_binlog=1` (meaningful when binary logging is enabled). MongoDB standalone uses acknowledged writes with **`WriteConcern(w=1, j=True)`**. Durability policies are closer but not identical, especially in terms of group commit and replication semantics.
- **Primary hypotheses pre-specified:** CRUD throughput differences at **20,000 records / 20 clients** (4 planned comparisons; Holm within this primary family); the other 56 comparisons, including JOIN/`$lookup`, are exploratory with a separate 60-test Holm correction. Do not pick different primary outcomes based on the observed results.
- **Resumable pair-level checkpoint:** a completed pair (both databases) is saved atomically to the output log; restart the experiment cell to continue from completed pairs. If a Colab runtime is recycled, local `/content/` files disappear unless Drive persistence was explicitly enabled.

### Existing methodology preserved
Numeric `Purchase` values, matched lookup indexes, MySQL database-enforced foreign keys (MongoDB application-managed references), equal deterministic plans, data reset, comparable SELECT/query results, measurement of response times and throughput, additional `multi_entity_read` workload. Single-node results do NOT establish horizontal scalability, fault tolerance, consistency superiority, or production behavior.

### How to run
1. Google Colab (Ubuntu x86_64, MySQL 8, MongoDB 8), upload `users.csv`, `products.csv`, `transactions_20000.csv`.
2. Run `RUN_MODE = 'smoke'` for sanity checking. No new full paper results yet.
3. Switch `RUN_MODE = 'full'`; optionally choose Drive persistence in the configuration cell; then Runtime → **Restart session and run all**.
4. **Full workload: 4 dataset sizes × 3 client settings × 5 operations × 2 DB × 15 repetitions = 1,800 timed runs** (900 paired configurations). This can take a long time; Colab time/resource limits vary.
5. Send the produced `ICIMCIS_revised_outputs_v3_full.zip` for audit; validate data and tables before changing the article.

**Do not mix** old `concurrency_results.csv` or the V2 smoke ZIP with V3 results.


## 1. Install real MySQL Server and MongoDB Server (not substitutes)

In [ ]:
%%bash
set -euo pipefail
. /etc/os-release
if [[ "$ID" != "ubuntu" || ("$VERSION_CODENAME" != "jammy" && "$VERSION_CODENAME" != "noble") ]]; then
  echo "Unsupported runtime: $ID / $VERSION_CODENAME. Use Ubuntu 22.04/24.04, do NOT install MariaDB as replacement." >&2
  exit 1
fi
if [[ "$(uname -m)" != "x86_64" ]]; then echo "This notebook expects x86_64 Colab" >&2; exit 1; fi
sudo apt-get update -qq
sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq mysql-server gnupg curl ca-certificates > /dev/null
sudo service mysql start
sudo mysql -Nse 'SELECT VERSION()' | sed 's/^/MySQL server: /'
if ! command -v mongod >/dev/null 2>&1; then
  curl -fsSL https://pgp.mongodb.com/server-8.0.asc | sudo gpg --dearmor --yes -o /usr/share/keyrings/mongodb-server-8.0.gpg
  echo "deb [ arch=amd64 signed-by=/usr/share/keyrings/mongodb-server-8.0.gpg ] https://repo.mongodb.org/apt/ubuntu ${VERSION_CODENAME}/mongodb-org/8.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-8.0.list > /dev/null
  sudo apt-get update -qq
  sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq mongodb-org > /dev/null
fi
sudo mkdir -p /data/db
sudo chown "$(id -u):$(id -g)" /data/db
if ! pgrep -x mongod > /dev/null; then
  mongod --fork --bind_ip 127.0.0.1 --dbpath /data/db --logpath /tmp/mongodb-benchmark.log
fi
mongod --version | head -n 1

**Installation notes:** Do not save DB credentials in public GitHub. Both servers run only on local Colab (`127.0.0.1`); MongoDB is unauthenticated in this *disposable local research VM*, **not safe for externally accessible servers**. If Colab image/package changes, stop and adapt using official installation docs; document installed server versions.

## 2. Python packages and experiment configuration

In [ ]:
%pip install -q pandas numpy scipy pymysql pymongo matplotlib

In [ ]:
from pathlib import Path
import os, sys, json, csv, hashlib, math, random, secrets, time, platform, subprocess, threading, zipfile
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from itertools import product
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import pymysql, pymongo
from pymongo import MongoClient, ASCENDING, DESCENDING
from pymongo.write_concern import WriteConcern

RUN_MODE = 'smoke'  # CHANGE TO 'full' after smoke test is successful
USE_GOOGLE_DRIVE_OUTPUT = False  # Optional: True to keep checkpoints across Colab runtime disconnections
BENCHMARK_PROTOCOL_ID = 'ICIMCIS_V3_JOURNALED_WRITES_15REPS'
MASTER_SEED = 20261015
OPS_PER_CLIENT = 20
VOLUMES = [1000,10000,15000,20000] if RUN_MODE == 'full' else [1000]
CLIENT_LEVELS = [5,10,20] if RUN_MODE == 'full' else [5]
OPERATIONS = ['insert','select','update','delete','multi_entity_read']
REPETITIONS = 15 if RUN_MODE == 'full' else 1
assert RUN_MODE in {'smoke','full'}
BASE_DIR = Path('/content') if Path('/content').exists() else Path.cwd()
if USE_GOOGLE_DRIVE_OUTPUT:
    if not (BASE_DIR / 'drive').exists():
        from google.colab import drive
        drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/ICIMCIS_2026_Benchmark')
else:
    OUTPUT_ROOT = BASE_DIR
OUT = OUTPUT_ROOT / f'icimcis_revised_outputs_v3_{RUN_MODE}'
OUT.mkdir(parents=True,exist_ok=True)
PLANNED_PAIRS = len(VOLUMES)*len(CLIENT_LEVELS)*len(OPERATIONS)*REPETITIONS
print('Mode:', RUN_MODE, '| planned timed runs:', 2*PLANNED_PAIRS)
print('Output / checkpoint folder:',OUT)
print('Durability settings will be asserted in the connection cell.')


## 3. Upload inputs and verify provenance / referential integrity

In [ ]:
required = ['users.csv','products.csv','transactions_20000.csv']
missing = [f for f in required if not (BASE_DIR / f).is_file()]
if missing:
    if 'google.colab' not in sys.modules:
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError(f'Missing: {missing}. Copy files to {BASE_DIR}')
    from google.colab import files
    print('Please upload missing files:', missing)
    files.upload()
assert all((BASE_DIR/f).exists() for f in required), 'Missing required dataset CSV files'

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as fp:
        for chunk in iter(lambda:fp.read(1024*1024), b''):h.update(chunk)
    return h.hexdigest()

users = pd.read_csv(BASE_DIR/'users.csv')
products = pd.read_csv(BASE_DIR/'products.csv')
trans = pd.read_csv(BASE_DIR/'transactions_20000.csv')
assert set(['User_ID','Gender','Age','Occupation','City_Category','Stay_In_Current_City_Years','Marital_Status']).issubset(users)
assert set(['Product_ID','Product_Category']).issubset(products)
assert set(['Transaction_ID','User_ID','Product_ID','Purchase']).issubset(trans)
assert len(trans)==20000 and trans['Transaction_ID'].is_unique
assert users['User_ID'].is_unique and products['Product_ID'].is_unique
assert not trans[['Transaction_ID','User_ID','Product_ID','Purchase']].isna().any().any()
assert set(trans.User_ID).issubset(set(users.User_ID))
assert set(trans.Product_ID).issubset(set(products.Product_ID))
assert (trans.Purchase >= 0).all()
trans=trans.sort_values('Transaction_ID').reset_index(drop=True)
assert trans.Transaction_ID.tolist()==list(range(1,20001)), 'Expected original sequential Transaction IDs'
for volume in [1000,10000,15000,20000]:
    optional = BASE_DIR/f'transactions_{volume}.csv'
    if optional.exists():
        old = pd.read_csv(optional)
        current = trans.head(volume).reset_index(drop=True)
        pd.testing.assert_frame_equal(old[current.columns].reset_index(drop=True),current,check_dtype=False)
print('Validated: users',len(users),'products',len(products),'transaction rows',len(trans))
input_hashes={f:sha256_file(BASE_DIR/f) for f in required}
if (BASE_DIR/'walmart.csv').is_file():
    input_hashes['walmart.csv']=sha256_file(BASE_DIR/'walmart.csv')
print('SHA-256 provenance:',json.dumps(input_hashes,indent=2))

## 4. Connections and durability verification

A successful `ping` is not proof of durable writes. Before benchmarking, this notebook explicitly sets MySQL InnoDB commit flushing to 1 and configures **MongoDB acknowledged, journaled writes** (`w=1`, `j=True`). MongoDB is a standalone server and does not demonstrate replica-set durability; write paths and group-commit policies still differ across engines. No unverified claim of *identical durability guarantees* should be made in the manuscript.


In [ ]:
# Disposable, local-only MySQL experiment account; never publish credentials.
BENCH_DATABASE = 'icimcis_bench_v3'
MYSQL_USER = 'icimcis_local'
MYSQL_PASSWORD = secrets.token_urlsafe(24)
admin_sql=(
    f"CREATE DATABASE IF NOT EXISTS `{BENCH_DATABASE}`;"
    f"CREATE USER IF NOT EXISTS '{MYSQL_USER}'@'localhost' IDENTIFIED BY '{MYSQL_PASSWORD}';"
    f"ALTER USER '{MYSQL_USER}'@'localhost' IDENTIFIED BY '{MYSQL_PASSWORD}';"
    f"GRANT ALL PRIVILEGES ON `{BENCH_DATABASE}`.* TO '{MYSQL_USER}'@'localhost';"
    "FLUSH PRIVILEGES;"
    "SET GLOBAL innodb_flush_log_at_trx_commit=1;"
    "SET GLOBAL sync_binlog=1;"
)
subprocess.run(['sudo','mysql','-e',admin_sql],check=True)
MYSQL_CFG=dict(host='127.0.0.1', port=3306,user=MYSQL_USER,password=MYSQL_PASSWORD,
               database=BENCH_DATABASE,charset='utf8mb4',autocommit=True,
               connect_timeout=15,read_timeout=120,write_timeout=120)
MONGO_URI='mongodb://127.0.0.1:27017'
MC=MongoClient(MONGO_URI,serverSelectionTimeoutMS=15000)
MC.admin.command('ping')
MONGO_WRITE_CONCERN = WriteConcern(w=1,j=True,wtimeout=60000)
DB=MC[BENCH_DATABASE].with_options(write_concern=MONGO_WRITE_CONCERN)
def sql_conn(): return pymysql.connect(**MYSQL_CFG)
def mongo_tx(): return DB['transactions']
with sql_conn() as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT VERSION()'); MYSQL_VERSION=cur.fetchone()[0]
        cur.execute('SELECT @@GLOBAL.innodb_flush_log_at_trx_commit, @@GLOBAL.sync_binlog, @@GLOBAL.log_bin')
        MYSQL_DURABILITY_TUPLE=cur.fetchone()
MONGO_VERSION=MC.server_info()['version']
assert 'MariaDB' not in MYSQL_VERSION, 'This study requires MySQL, not MariaDB'
assert int(MYSQL_DURABILITY_TUPLE[0])==1, 'MySQL innodb_flush_log_at_trx_commit must be 1'
assert int(MYSQL_DURABILITY_TUPLE[1])==1, 'MySQL sync_binlog must be 1'
assert DB.write_concern.document.get('j') is True and DB.write_concern.document.get('w')==1
MONGO_SERVER_STATUS=MC.admin.command('serverStatus')
MONGO_STORAGE_ENGINE=MONGO_SERVER_STATUS.get('storageEngine',{})
assert MONGO_STORAGE_ENGINE.get('persistent') is True, 'Expected persistent MongoDB storage engine'
DURABILITY_CONFIG={
    'mysql_innodb_flush_log_at_trx_commit':int(MYSQL_DURABILITY_TUPLE[0]),
    'mysql_sync_binlog':int(MYSQL_DURABILITY_TUPLE[1]),
    'mysql_log_bin_enabled':bool(int(MYSQL_DURABILITY_TUPLE[2])),
    'mongodb_write_concern':DB.write_concern.document,
    'mongodb_storage_engine':MONGO_STORAGE_ENGINE.get('name'),
    'mongodb_storage_engine_persistent':MONGO_STORAGE_ENGINE.get('persistent'),
    'mongodb_standalone':True,
    'note':'Durability settings explicitly documented, but SQL and Mongo have non-identical internals / commit paths.'
}
print('Server engines:',MYSQL_VERSION,'| MongoDB', MONGO_VERSION)
print('Verified durability:',json.dumps(DURABILITY_CONFIG,indent=2,default=str))


### Intentional schema difference
We enforce original Users/Products foreign keys on MySQL, but MongoDB stores matching references without database-enforced foreign keys. Therefore this benchmark compares actual implementations and constraints, not identical transaction semantics. Include this explicitly in paper. The `Transaction_ID` lookup index and secondary transaction `User_ID`/`Product_ID` indexes are matched (MySQL requires indexes for its FKs); Users and Products also get indexed identifiers for the multi-entity read. No extra Purchase index.

In [ ]:
with sql_conn() as conn:
    with conn.cursor() as cur:
        cur.execute('DROP TABLE IF EXISTS transactions')
        cur.execute('DROP TABLE IF EXISTS products')
        cur.execute('DROP TABLE IF EXISTS users')
        cur.execute("""CREATE TABLE users(
            User_ID INT PRIMARY KEY, Gender VARCHAR(1), Age VARCHAR(8), Occupation INT,
            City_Category VARCHAR(1), Stay_In_Current_City_Years VARCHAR(3), Marital_Status INT
        ) ENGINE=InnoDB""")
        cur.execute("""CREATE TABLE products(
            Product_ID VARCHAR(16) PRIMARY KEY, Product_Category INT
        ) ENGINE=InnoDB""")
        cur.execute("""CREATE TABLE transactions(
            Transaction_ID INT PRIMARY KEY,
            User_ID INT NOT NULL,
            Product_ID VARCHAR(16) NOT NULL,
            Purchase DECIMAL(10,2) NOT NULL,
            CONSTRAINT fk_user FOREIGN KEY(User_ID) REFERENCES users(User_ID),
            CONSTRAINT fk_product FOREIGN KEY(Product_ID) REFERENCES products(Product_ID)
        ) ENGINE=InnoDB""")
        cur.executemany("INSERT INTO users VALUES (%s,%s,%s,%s,%s,%s,%s)",
            [tuple(r) for r in users[['User_ID','Gender','Age','Occupation','City_Category','Stay_In_Current_City_Years','Marital_Status']].itertuples(index=False,name=None)])
        cur.executemany('INSERT INTO products VALUES (%s,%s)',
            [tuple(r) for r in products[['Product_ID','Product_Category']].itertuples(index=False,name=None)])

# Rebuild ONLY this isolated MongoDB database (destructive here by design)
MC.drop_database(BENCH_DATABASE)
DB=MC[BENCH_DATABASE].with_options(write_concern=MONGO_WRITE_CONCERN)
DB['users'].insert_many([{'User_ID':int(r.User_ID),'Gender':str(r.Gender),'Age':str(r.Age),
  'Occupation':int(r.Occupation),'City_Category':str(r.City_Category),
  'Stay_In_Current_City_Years':str(r.Stay_In_Current_City_Years),
  'Marital_Status':int(r.Marital_Status)} for r in users.itertuples(index=False)])
DB['products'].insert_many([{'Product_ID':str(r.Product_ID),'Product_Category':int(r.Product_Category)}
   for r in products.itertuples(index=False)])
DB['users'].create_index([('User_ID',ASCENDING)], unique=True)
DB['products'].create_index([('Product_ID',ASCENDING)], unique=True)
DB['transactions'].create_index([('Transaction_ID',ASCENDING)], unique=True)
# InnoDB automatically indexes MySQL child foreign-key columns; mirror those indexes here.
DB['transactions'].create_index([('User_ID',ASCENDING)])
DB['transactions'].create_index([('Product_ID',ASCENDING)])
assert DB['users'].count_documents({})==len(users)
assert DB['products'].count_documents({})==len(products)
print('Schema ready: 3 entities in both databases; transaction index matched.')

## 5. Deterministic data restoration and operation plans

In [ ]:
FIELDS=['Transaction_ID','User_ID','Product_ID','Purchase']
def volume_df(n): return trans.iloc[:n][FIELDS].copy()
def mysql_reset(df):
    rows=[(int(t),int(u),str(p),int(v)) for t,u,p,v in df.itertuples(index=False,name=None)]
    with sql_conn() as conn:
        with conn.cursor() as cur:
            cur.execute('DELETE FROM transactions')  # FK-safe; can be slower than TRUNCATE, not timed
            cur.executemany('INSERT INTO transactions (Transaction_ID,User_ID,Product_ID,Purchase) VALUES (%s,%s,%s,%s)',rows)

def mongo_reset(df):
    coll=mongo_tx()
    coll.delete_many({})
    docs=[{'Transaction_ID':int(t),'User_ID':int(u),'Product_ID':str(p),'Purchase':int(v)}
          for t,u,p,v in df.itertuples(index=False,name=None)]
    if docs:coll.insert_many(docs,ordered=False)

def verify_loaded(db,volume):
    if db=='mysql':
        with sql_conn() as conn:
            with conn.cursor() as cur:
                cur.execute('SELECT COUNT(*) FROM transactions'); n=cur.fetchone()[0]
    else:n=mongo_tx().count_documents({})
    assert n==volume, (db,volume,n)

def warmup_indexed_reads(db, volume, requests=5):
    """Equal, unmeasured, non-mutating warm-up probes before each timed trial."""
    ids=list(range(1,min(requests,volume)+1))
    if db=='mysql':
        with sql_conn() as conn:
            with conn.cursor() as cur:
                for tid in ids:
                    cur.execute('SELECT Transaction_ID FROM transactions WHERE Transaction_ID=%s',(tid,))
                    assert cur.fetchone()[0]==tid
    else:
        coll=mongo_tx()
        for tid in ids:
            rec=coll.find_one({'Transaction_ID':tid},{'_id':0,'Transaction_ID':1})
            assert rec['Transaction_ID']==tid

def make_plan(df,n_clients,rep,operation):
    # SAME deterministic schedule passed to BOTH systems: no per-worker RNG or overlap
    seed=MASTER_SEED+len(df)*1000+n_clients*100+rep*7+sum(ord(c) for c in operation)
    rng=random.Random(seed)
    plan=[]
    for cid in range(n_clients):
        client_ops=[]
        for i in range(OPS_PER_CLIENT):
            logical_idx=cid*OPS_PER_CLIENT+i
            if operation=='insert':
                source=df.iloc[rng.randrange(len(df))]
                client_ops.append((9_000_000+logical_idx,int(source.User_ID),str(source.Product_ID),
                                   rng.randrange(100,25000)))
            elif operation in {'update','delete'}:
                client_ops.append(1+logical_idx)  # each client a disjoint ID, exists in every volume
            else:
                client_ops.append(rng.randrange(5000,18001))
        plan.append(client_ops)
    assert n_clients*OPS_PER_CLIENT<=len(df)
    return plan

## 6. Equivalent SQL / MongoDB operations, barrier, and per-operation measurements

In [ ]:
SELECT_SQL=('SELECT Transaction_ID,User_ID,Product_ID,Purchase FROM transactions '
 'WHERE Purchase > %s ORDER BY Purchase DESC,Transaction_ID ASC LIMIT 200')
JOIN_SQL=('SELECT t.Transaction_ID,t.Purchase,u.Gender,p.Product_Category '
 'FROM transactions t JOIN users u ON t.User_ID=u.User_ID '
 'JOIN products p ON t.Product_ID=p.Product_ID '
 'WHERE t.Purchase > %s ORDER BY t.Purchase DESC,t.Transaction_ID ASC LIMIT 200')

def sql_step(cur,operation,task):
    if operation=='insert':
        cur.execute('INSERT INTO transactions VALUES (%s,%s,%s,%s)',task)
        return cur.rowcount
    if operation=='update':
        cur.execute('UPDATE transactions SET Purchase=Purchase+1 WHERE Transaction_ID=%s',(task,))
        return cur.rowcount
    if operation=='delete':
        cur.execute('DELETE FROM transactions WHERE Transaction_ID=%s',(task,))
        return cur.rowcount
    if operation in {'select','multi_entity_read'}:
        cur.execute(JOIN_SQL if operation=='multi_entity_read' else SELECT_SQL,(task,))
        return len(cur.fetchall())
    raise ValueError(operation)

def mongo_step(coll,operation,task):
    if operation=='insert':
        t,u,p,v=task
        coll.insert_one({'Transaction_ID':t,'User_ID':u,'Product_ID':p,'Purchase':v})
        return 1
    if operation=='update':
        r=coll.update_one({'Transaction_ID':task},{'$inc':{'Purchase':1}})
        assert r.matched_count==1
        return r.modified_count
    if operation=='delete':
        return coll.delete_one({'Transaction_ID':task}).deleted_count
    if operation=='select':
        cursor=coll.find({'Purchase':{'$gt':task}}, {'_id':0,'Transaction_ID':1,'User_ID':1,'Product_ID':1,'Purchase':1})
        return len(list(cursor.sort([('Purchase',DESCENDING),('Transaction_ID',ASCENDING)]).limit(200)))
    if operation=='multi_entity_read':
        pipeline=[
          {'$match':{'Purchase':{'$gt':task}}},
          {'$sort':{'Purchase':-1,'Transaction_ID':1}},
          {'$limit':200},
          {'$lookup':{'from':'users','localField':'User_ID','foreignField':'User_ID','as':'u'}},
          {'$unwind':'$u'},
          {'$lookup':{'from':'products','localField':'Product_ID','foreignField':'Product_ID','as':'p'}},
          {'$unwind':'$p'},
          {'$project':{'_id':0,'Transaction_ID':1,'Purchase':1,'Gender':'$u.Gender','Product_Category':'$p.Product_Category'}}
        ]
        return len(list(coll.aggregate(pipeline)))
    raise ValueError(operation)

def run_workers(db,operation,plan,volume,rep):
    n_clients=len(plan)
    barrier=threading.Barrier(n_clients+1,timeout=120)
    def worker(cid):
        if db=='mysql':
            conn=sql_conn(); cur=conn.cursor(); coll=None
        else:
            conn=None;cur=None;coll=mongo_tx()
        per_op=[]
        try:
            barrier.wait()
            for idx,task in enumerate(plan[cid]):
                start=time.perf_counter_ns()
                affected=sql_step(cur,operation,task) if db=='mysql' else mongo_step(coll,operation,task)
                latency_ms=(time.perf_counter_ns()-start)/1e6
                if operation in {'insert','update','delete'}:assert affected==1, (db,operation,cid,idx,affected)
                per_op.append((cid,idx,latency_ms,int(affected)))
        finally:
            if cur is not None:cur.close()
            if conn is not None:conn.close()
        return per_op
    with ThreadPoolExecutor(max_workers=n_clients) as pool:
        jobs=[pool.submit(worker,cid) for cid in range(n_clients)]
        start=time.perf_counter()
        barrier.wait()  # releases clients together; connection creation is outside timed operations
        per_ops=[item for j in jobs for item in j.result()]
        elapsed=time.perf_counter()-start
    assert len(per_ops)==n_clients*OPS_PER_CLIENT
    vals=[r[2] for r in per_ops]
    expected=volume+(n_clients*OPS_PER_CLIENT if operation=='insert' else
                     -n_clients*OPS_PER_CLIENT if operation=='delete' else 0)
    verify_loaded(db,expected)
    run_row=dict(data_volume=volume,database=db,operation=operation,concurrency_clients=n_clients,
            replicate=rep,ops_per_client=OPS_PER_CLIENT,total_ops=len(vals),
            avg_response_ms=float(np.mean(vals)),median_response_ms=float(np.median(vals)),
            p95_response_ms=float(np.percentile(vals,95)),
            total_wall_time_sec=elapsed,throughput_ops_sec=len(vals)/elapsed,
            result_rows_total=sum(x[3] for x in per_ops),run_mode=RUN_MODE)
    op_rows=[dict(data_volume=volume,database=db,operation=operation,concurrency_clients=n_clients,
                 replicate=rep,client_id=cid,operation_idx=idx,latency_ms=ms,result_rows=aff,
                 run_mode=RUN_MODE) for cid,idx,ms,aff in per_ops]
    return run_row,op_rows

**Important validity distinctions:** `multi_entity_read` measures joined data access with both sides indexed on User_ID and Product_ID. SQL uses actual joins and MySQL FK constraints; Mongo uses `$lookup` and application-managed references. INSERT is one acknowledged request at a time; there is **no batch insert inside timing**. The timing of SELECT includes all returned rows; p95 of individual requests is **not** a 95% confidence interval.

## 7. Paired full experiment with safe resume/checkpointing

Each paired configuration is recorded only after **both** DB trials succeed and their operation-level logs have been collected. If a cell fails or you interrupt execution, **rerun this experiment cell**. It skips valid completed pairs; never mixes results from different protocol versions, data files, engines, or durability settings. The checksum metadata ensures the checkpoint belongs to the current configuration.

For long Colab jobs, consider `USE_GOOGLE_DRIVE_OUTPUT=True` and rerun the notebook. Drive storage is slower; keep enough quota available. `/content` alone does not survive runtime replacement. A checkpoint is an aid, not a substitute for a completed full experiment.


In [ ]:
RUN_FILE=OUT/'benchmark_runs.csv'
OPS_FILE=OUT/'operation_latencies.csv'
CONFIG_FILE=OUT/'checkpoint_configuration.json'
config={
  'protocol_id':BENCHMARK_PROTOCOL_ID,'run_mode':RUN_MODE,
  'mysql_version':MYSQL_VERSION,'mongo_version':MONGO_VERSION,
  'durability':DURABILITY_CONFIG,
  'input_sha256':input_hashes,'volumes':VOLUMES,'clients':CLIENT_LEVELS,
  'operations':OPERATIONS,'repetitions':REPETITIONS,
  'seed':MASTER_SEED,'ops_per_client':OPS_PER_CLIENT,
  'dataset_source':'original e-Commerce (Walmart) dataset, first-N rows; see paper'
}
# Consistency of all settings, not just the number of repetitions, is mandatory for resume.
config_fingerprint=hashlib.sha256(json.dumps(config,sort_keys=True,default=str).encode()).hexdigest()
if CONFIG_FILE.exists():
    existing=json.loads(CONFIG_FILE.read_text())
    if existing.get('config_sha256')!=config_fingerprint:
        raise RuntimeError('Checkpoint does not match data/server/method configuration. '
              'Use a fresh output directory; do not merge incompatible runs.')
else:
    if RUN_FILE.exists() or OPS_FILE.exists():
        raise RuntimeError('Existing CSV without checkpoint manifest. Move them aside; do not merge old results.')
    CONFIG_FILE.write_text(json.dumps({'config_sha256':config_fingerprint,'config':config},indent=2,default=str))

PAIR_KEYS=['data_volume','concurrency_clients','operation','replicate']
KEY_DB=PAIR_KEYS+['database']
def read_if_exists(path):
    return pd.read_csv(path) if path.exists() and path.stat().st_size else pd.DataFrame()

def clean_existing_pairs():
    a=read_if_exists(RUN_FILE)
    b=read_if_exists(OPS_FILE)
    if a.empty and b.empty:return set()
    if a.empty or b.empty:
        print('Repairing an incomplete checkpoint by rerunning its trials')
        a=pd.DataFrame();b=pd.DataFrame()
    else:
        run_cnt=a.groupby(KEY_DB).size()
        op_cnt=b.groupby(KEY_DB).size()
        good=[]
        for key,count in run_cnt.items():
            expected=int(key[1])*OPS_PER_CLIENT
            if count==1 and op_cnt.get(key,0)==expected:
                good.append(key)
        gooddb=set(good)
        good_pairs=set()
        for key in gooddb:
            other=tuple(key[:-1])+('mongodb' if key[-1]=='mysql' else 'mysql',)
            if other in gooddb:good_pairs.add(tuple(key[:-1]))
        def keep(frame):
            if frame.empty:return frame
            k=[tuple(row) for row in frame[PAIR_KEYS].itertuples(index=False,name=None)]
            return frame.loc[[t in good_pairs for t in k]].copy()
        ra=keep(a);rb=keep(b)
        if len(ra)!=len(a) or len(rb)!=len(b):
            print('Removed incomplete paired checkpoint rows and will rerun them.')
            if not ra.empty:ra.to_csv(RUN_FILE,index=False)
            elif RUN_FILE.exists():RUN_FILE.unlink()
            if not rb.empty:rb.to_csv(OPS_FILE,index=False)
            elif OPS_FILE.exists():OPS_FILE.unlink()
        return good_pairs
    for f in (RUN_FILE,OPS_FILE):
        if f.exists():f.unlink()
    return set()

completed=clean_existing_pairs()
print(f'Resuming: {len(completed)} / {PLANNED_PAIRS} paired configurations complete.')
new_pairs=0
for volume,clients,operation,rep in product(VOLUMES,CLIENT_LEVELS,OPERATIONS,range(REPETITIONS)):
    pair=(volume,clients,operation,rep)
    if pair in completed:continue
    df=volume_df(volume)
    plan=make_plan(df,clients,rep,operation)
    db_order=['mysql','mongodb'] if (rep+clients+volume+len(operation))%2==0 else ['mongodb','mysql']
    pair_runs=[];pair_ops=[]
    for db in db_order:
        (mysql_reset if db=='mysql' else mongo_reset)(df)
        verify_loaded(db,volume)
        warmup_indexed_reads(db,volume,requests=5)
        r,operation_rows=run_workers(db,operation,plan,volume,rep)
        r['pair_order']=db_order.index(db)+1
        pair_runs.append(r)
        pair_ops.extend(operation_rows)
    assert len(pair_runs)==2 and len(pair_ops)==2*clients*OPS_PER_CLIENT
    # Append only after BOTH trials complete. If interrupted while writing, startup repair removes incomplete pairs.
    pd.DataFrame(pair_runs).to_csv(RUN_FILE,mode='a',header=not RUN_FILE.exists(),index=False)
    pd.DataFrame(pair_ops).to_csv(OPS_FILE,mode='a',header=not OPS_FILE.exists(),index=False)
    completed.add(pair)
    new_pairs+=1
    if new_pairs==1 or new_pairs%10==0 or len(completed)==PLANNED_PAIRS:
        print(f'Completed {len(completed)}/{PLANNED_PAIRS} paired configurations (new this execution: {new_pairs})')

raw_results=pd.read_csv(RUN_FILE)
assert len(raw_results)==2*PLANNED_PAIRS, f'Incomplete run: expected {2*PLANNED_PAIRS}, got {len(raw_results)}'
assert raw_results[KEY_DB].duplicated().sum()==0
assert len(pd.read_csv(OPS_FILE))==(sum(2*c*OPS_PER_CLIENT*REPETITIONS for c in CLIENT_LEVELS)*len(VOLUMES)*len(OPERATIONS))
print('ALL COMPLETE:',len(raw_results),'run-level observations, with operation-level data in',OPS_FILE)


## 8. Descriptive analysis, 95% paired bootstrap intervals, and correction for multiple comparisons

The **4 pre-defined primary hypotheses** are CRUD at 20,000 transactions and 20 concurrent clients. We apply Holm to those 4 tests; the full set of 60 workload combinations is exploratory and separately receives Holm correction. P-values are not proof of practical usefulness, and dependent trials on a shared VM may weaken classical inferential assumptions. Bootstrap intervals are pointwise (not familywise simultaneous confidence bands).


In [ ]:
summary=(raw_results.groupby(['data_volume','concurrency_clients','operation','database'])
     .agg(n_runs=('replicate','count'),throughput_mean=('throughput_ops_sec','mean'),
          throughput_sd=('throughput_ops_sec','std'),
          median_request_ms_mean=('median_response_ms','mean'),
          p95_request_ms_mean=('p95_response_ms','mean'))
     .reset_index())
summary.to_csv(OUT/'descriptive_summary.csv',index=False)
print('Mean throughput and request latency summaries:')
print(summary.to_string(index=False))

paired=raw_results.pivot(index=['data_volume','concurrency_clients','operation','replicate'],
                         columns='database',values='throughput_ops_sec').dropna().reset_index()
paired['ratio_mongo_over_mysql']=paired['mongodb']/paired['mysql']

def bootstrap_geomean_ratio(ratios,seed,n_boot=5000):
    ratios=np.asarray(ratios,dtype=float)
    assert np.all(np.isfinite(ratios)) and np.all(ratios>0)
    logs=np.log(ratios)
    rng=np.random.default_rng(seed)
    draws=rng.choice(logs,size=(n_boot,len(logs)),replace=True)
    lo,hi=np.percentile(np.exp(draws.mean(axis=1)),[2.5,97.5])
    return float(np.exp(logs.mean())),float(lo),float(hi)

def holm_adjust(pvalues):
    p=np.asarray(pvalues,dtype=float)
    order=np.argsort(p)
    corrected=np.empty_like(p)
    running=0.0
    for rank,pos in enumerate(order):
        running=max(running,(len(p)-rank)*p[pos])
        corrected[pos]=min(1.0,running)
    return corrected

comparison=[]
for (vol,clients,op),g in paired.groupby(['data_volume','concurrency_clients','operation']):
    x=g['ratio_mongo_over_mysql'].to_numpy(dtype=float)
    n=len(x)
    is_primary=(vol==20000 and clients==20 and op in {'insert','select','update','delete'})
    if n>=5:
        ratio,lo,hi=bootstrap_geomean_ratio(x,MASTER_SEED+vol+clients+len(op))
        d=np.log(x)
        pval=float(stats.wilcoxon(d,zero_method='wilcox',method='auto').pvalue) if np.any(d!=0) else 1.0
    else:
        ratio=float(np.exp(np.log(x).mean()))
        lo=hi=pval=float('nan')
    comparison.append(dict(data_volume=vol,concurrency_clients=clients,operation=op,
                           n_paired_runs=n,is_primary_hypothesis=is_primary,
                           geometric_ratio_mongo_over_mysql=ratio,
                           ratio_ci95_low=lo,ratio_ci95_high=hi,wilcoxon_p=pval))
comparison=pd.DataFrame(comparison)
comparison['holm_all_60_p']=np.nan
valid=comparison['wilcoxon_p'].notna()
if valid.any():
    comparison.loc[valid,'holm_all_60_p']=holm_adjust(comparison.loc[valid,'wilcoxon_p'])
comparison['holm_primary_4_p']=np.nan
primary_mask=valid & comparison['is_primary_hypothesis']
if primary_mask.any():
    comparison.loc[primary_mask,'holm_primary_4_p']=holm_adjust(comparison.loc[primary_mask,'wilcoxon_p'])
comparison.to_csv(OUT/'paired_statistical_comparison.csv',index=False)
print('Paired throughput ratio MongoDB/MySQL (larger than 1 means MongoDB higher throughput):')
print(comparison.to_string(index=False))
if RUN_MODE=='smoke':
    print('SMOKE MODE: n=1 and ALL p-values are NA. Do not report paper significance.')
else:
    assert (comparison.n_paired_runs==REPETITIONS).all()
    print('Report effect sizes + uncertainty; distinguish primary family (4) from exploratory family (60).')


## 9. Reviewer-ready figures (ONLY full-mode validated results)

Generate separate figures by operation: throughput vs data volume at 20 clients, and p95 request latency vs client count at 20,000 records. Figures are descriptive; include n and uncertainty in the paper's caption. Use only after full execution.

In [ ]:
import matplotlib.pyplot as plt
if RUN_MODE != 'full':
    print('Skipping paper plots: smoke mode is not a study result.')
else:
    for op in OPERATIONS:
        s=summary[(summary['operation']==op)&(summary['concurrency_clients']==20)]
        fig,ax=plt.subplots(figsize=(7,4.5))
        for db in ['mysql','mongodb']:
            part=s[s['database']==db].sort_values('data_volume')
            ax.errorbar(part['data_volume'],part['throughput_mean'],yerr=part['throughput_sd'],
                        marker='o',capsize=3,label=db)
        ax.set(xlabel='Number of transactions',ylabel='Throughput (operations/second)',
               title=f'{op} — 20 concurrent clients ({REPETITIONS} replicates)')
        ax.legend();fig.tight_layout()
        fig.savefig(OUT/f'figure_throughput_{op}_20clients.png',dpi=180)
        plt.close(fig)
    for op in OPERATIONS:
        s=summary[(summary['operation']==op)&(summary['data_volume']==20000)]
        fig,ax=plt.subplots(figsize=(7,4.5))
        for db in ['mysql','mongodb']:
            part=s[s['database']==db].sort_values('concurrency_clients')
            ax.plot(part['concurrency_clients'],part['p95_request_ms_mean'],marker='o',label=db)
        ax.set(xlabel='Concurrent clients',ylabel='Mean of per-run p95 request latency (ms)',
               title=f'{op} — 20,000 transactions ({REPETITIONS} replicates)')
        ax.legend();fig.tight_layout()
        fig.savefig(OUT/f'figure_p95_{op}_20000rows.png',dpi=180)
        plt.close(fig)
    print('Exploratory plots saved. Verify axes and captions before inclusion in manuscript.')


## 10. Reproducibility manifest, checksums, outputs ZIP, and optional download

In [ ]:
meta={
 'study':'ICIMCIS 2026 MySQL vs MongoDB revised concurrent benchmark V3',
 'protocol_id':BENCHMARK_PROTOCOL_ID,
 'generated_at_utc':datetime.now(timezone.utc).isoformat(),
 'run_mode':RUN_MODE,'mysql_version':MYSQL_VERSION,'mongodb_version':MONGO_VERSION,
 'mysql_write_settings_and_mongo_write_concern':DURABILITY_CONFIG,
 'pandas_version':pd.__version__,'numpy_version':np.__version__,'scipy_version':scipy.__version__,
 'python_version':sys.version,'platform':platform.platform(),
 'os_release':Path('/etc/os-release').read_text() if Path('/etc/os-release').exists() else 'unknown',
 'cpu_count':os.cpu_count(),'seed':MASTER_SEED,'volumes':VOLUMES,'clients':CLIENT_LEVELS,
 'operations':OPERATIONS,'repetitions':REPETITIONS,'ops_per_client':OPS_PER_CLIENT,
 'planned_timed_runs':2*PLANNED_PAIRS,
 'primary_hypotheses':'CRUD throughput: 20000 rows, 20 clients, Holm 4; all 60 combos exploratory Holm 60',
 'schema_note':'MySQL FK-enforced InnoDB; Mongo document references without DB FKs; comparable query lookup indexes',
 'workload_note':'Deterministic plans, reset before each DB trial, five PK read warmups, alternating DB order',
 'dataset_source_citation':'V. Devaraj, e-Commerce (Walmart) Sales Dataset, v1 May 2024; access/license verify independently',
 'dataset_original_url':'https://www.kaggle.com/datasets/devarajv88/walmart-sales-dataset',
 'input_sha256':input_hashes,
 'output_columns':list(raw_results.columns)
}
(OUT/'experiment_manifest.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False,default=str))
(OUT/'READ_ME_FIRST.txt').write_text(
 f'V3 experiment results, mode={RUN_MODE}. Smoke is NOT publishable.\n'
 'MySQL flush=1/sync_binlog=1 and Mongo WriteConcern(w=1,j=True) are logged in manifest.\n'
 'Do not merge previous paper/concurrent CSV or V2 smoke results with these files.\n'
 'Primary comparisons are only CRUD at 20000 records and 20 concurrent clients.\n'
 'Raw Walmart dataset redistribution requires confirmation of license/permission.\n',encoding='utf-8')
zip_path=OUTPUT_ROOT/f'ICIMCIS_revised_outputs_v3_{RUN_MODE}.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zz:
    for f in OUT.iterdir():
        if f.is_file():zz.write(f,arcname=f.name)
print('ZIP ready:',zip_path)
print('Files:', [f.name for f in OUT.iterdir() if f.is_file()])
# If in Colab and output saved locally, manually download:
# from google.colab import files
# files.download(str(zip_path))


## 11. What belongs in the revised paper (only after the full run)

1. `experiment_manifest.json` must show `run_mode=full`, `repetitions=15`, and the verified MySQL/MongoDB write-durability parameters.
2. Report 15 run-level means ± SD per configuration, throughput, response-time p95, and 95% paired bootstrap intervals (not a claim of 15 independent database installations).
3. For the 4 *pre-specified* CRUD primary hypotheses at **20,000 rows / 20 clients**, use `holm_primary_4_p`; the exploratory set of 60 conditions uses `holm_all_60_p`. The pointwise 95% bootstrap intervals are not themselves adjusted for multiplicity.
4. Do not imply MongoDB journaling `j=True` and InnoDB flush behavior are internally identical. MongoDB has no DB-enforced FKs here; identify this architectural asymmetry and its effect on interpretation.
5. Keep original local-machine sequential results separate. Do not merge V2 smoke/older concurrency results with V3 results or compare absolute latencies across machines.
6. `multi_entity_read` is new and tests query pattern differences, not a production checkout transaction. Respect hardware, scale (≤20,000 transactions), concurrency (≤20 clients), and single-node limitations.
7. If the Colab run was interrupted, resume via the experiment cell with the SAME dataset, server versions and config (persist Drive outputs if runtime might reset). The notebook will reject incompatible checkpoints.
8. Public dataset redistribution: confirm source license or explicit permission first; source page accessibility and license status must be reported accurately.
